In [ ]:
#| default_exp handlers.pipeline.transformer

# Transformer

`build_chain(cfg)` — assembles a fully data-driven, zero-runtime-if Callback chain from a `HandlerConfig`.

In [ ]:
#| export
from __future__ import annotations
from marisco.callbacks import (
    RenameColsCB, SoftParseDateTimeCB, SoftMeltWideNuclidesCB,
    SoftConvertUnitCB, SoftRemapCB, SanitizeLonLatCB, EncodeTimeCB, AddSampleIDCB,
)
from marisco.handlers.pipeline.loader import HandlerConfig

## build_chain

Assembles the standard 11-CB pipeline from a `HandlerConfig`.
No `if` at runtime — empty YAML lists produce Null-Object no-op CBs.

In [ ]:
#| export
def build_chain(cfg: HandlerConfig) -> list:
    "Assemble a data-driven Callback chain; empty YAML fields produce Null-Object no-op CBs."
    col_provider = next((v for v in cfg.rename.values() if v.endswith('_PROVIDER')), None)
    return [
        RenameColsCB(mapping=cfg.rename, string_cast=cfg.string_cast),
        SoftParseDateTimeCB(col_date=cfg.col_date, col_time=cfg.col_time, fmt=cfg.dt_format),
        SoftMeltWideNuclidesCB(spec=[s.model_dump() for s in cfg.melt_spec]),
        *[SoftConvertUnitCB(rule=r.model_dump()) for r in cfg.unit_conversions],
        SoftRemapCB(col_src='NUCLIDE', col_remap='NUCLIDE', lut=cfg.nuclide_lut),
        SoftRemapCB(col_src='UNIT',    col_remap='UNIT',    lut=cfg.unit_lut),
        SoftRemapCB(col_src='LAB',     col_remap='LAB',     lut=cfg.lab_lut),
        SoftRemapCB(col_src='NUCLIDE', col_remap='AREA',    lut={}, default_val=cfg.area_default),
        SanitizeLonLatCB(),
        EncodeTimeCB(),
        AddSampleIDCB(col_provider=col_provider),
    ]

In [ ]:
cfg   = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
chain = build_chain(cfg)

print(f"Chain length: {len(chain)}")
for i, cb in enumerate(chain, 1):
    print(f"  {i:2d}. {type(cb).__name__}")

assert len(chain) >= 10, f"Expected >= 10 CBs, got {len(chain)}"
assert type(chain[0]).__name__ == "RenameColsCB"
assert type(chain[-1]).__name__ == "AddSampleIDCB"
print("\nbuild_chain(fram_strait) ✓ — all assertions passed")